In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
import os
import warnings
import itertools

warnings.filterwarnings('ignore')

# ۱. تنظیمات مسیرها و متغیرها
file_path = r'second_stage_inputs\G11\dsas_g11_bearings_vibration_temp_output.xlsx'
output_filename4 = r'outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output1.xlsx'

os.makedirs(os.path.dirname(output_filename4), exist_ok=True)

all_features = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
                'AssetID_9368', 'AssetID_9369', 'AssetID_9370', 'AssetID_12208',
                'AssetID_9343', 'AssetID_9344', 'AssetID_9408']

target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']

# ۲. خواندن و پیش‌پردازش
df_raw = pd.read_excel(file_path)
df_raw['date'] = pd.to_datetime(df_raw['date'])
df_raw = df_raw.sort_values(by='date')

# ۳. حذف ۱۰ درصد داده‌های پرت با DBSCAN (حفظ منطق قبلی)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df_raw[all_features])
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(scaled_data)

# محاسبه فاصله‌ها برای حذف ۱۰ درصد پرت
from scipy.spatial.distance import cdist
cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}
def calc_dist(i):
    label, point = labels[i], scaled_data[i].reshape(1, -1)
    if label != -1: return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
    return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

df_raw['distance'] = [calc_dist(i) for i in range(len(df_raw))]
df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)  0.1):].copy()
df_cleaned = df_cleaned.sort_values(by='date').set_index('date')

# ۴. جداسازی بازه‌ها (نرمال و یک ماه آخر برای تست)
last_date = df_cleaned.index.max()
start_analysis_date = last_date - pd.Timedelta(days=30)
baseline_end = start_analysis_date - pd.Timedelta(days=1)
baseline_start = baseline_end - pd.Timedelta(days=30)

df_baseline = df_cleaned[(df_cleaned.index >= baseline_start) & (df_cleaned.index <= baseline_end)]
df_fault = df_cleaned[df_cleaned.index >= start_analysis_date].copy()

# ۵. تست ۳ برابر انحراف معیار (3-Sigma Rule)
for col in target_sensors:
    m = df_baseline[col].mean()
    s = df_baseline[col].std()
    upper = m + 3*s
    lower = m - 3*s
    # ایجاد ستون وضعیت برای هر سنسور تارگت
    df_fault[f'Status_{col}'] = df_fault[col].apply(lambda x: 'Fault' if (x > upper or x < lower) else 'Normal')

# ۶. تابع همبستگی جزئی
def partial_correlation_matrix(data, columns):
    corr = data[columns].corr().values
    precision = np.linalg.inv(corr + np.eye(corr.shape[0]) * 1e-6)
    d = np.sqrt(np.diag(precision))
    partial_corr = -precision / np.outer(d, d)
    np.fill_diagonal(partial_corr, 1.0)
    return pd.DataFrame(partial_corr, index=columns, columns=columns)

# ۷. محاسبات RCA
pcorr_baseline = partial_correlation_matrix(df_baseline, all_features)
pcorr_fault = partial_correlation_matrix(df_fault[all_features], all_features)
delta_pcorr = pcorr_fault - pcorr_baseline


# محاسبه شاخص‌ها
deviation_scores = {c: abs((df_fault[c].mean() - df_baseline[c].mean()) / (df_baseline[c].std() if df_baseline[c].std() !=0 else 1e-6)) for c in all_features}
change_scores = {c: np.mean([abs(delta_pcorr.loc[c, o]) for o in all_features if o != c]) for c in all_features}

# نرمال‌سازی و امتیاز نهایی
def norm(d):
    v = np.array(list(d.values()))
    return {k: (v_i - v.min()) / (v.max() - v.min() + 1e-6) for k, v_i in d.items()}

dev_n, chg_n = norm(deviation_scores), norm(change_scores)
rca_scores = {c: (0.5*dev_n[c] + 0.5*chg_n[c]) for c in all_features}

# ۸. آماده‌سازی خروجی نهایی
rca_summary = pd.DataFrame({
    'Sensor': all_features,
    'RCA_Score': [rca_scores[c] for c in all_features],
    'Suspicion': ['HIGH' if rca_scores[c] > 0.7 else 'MEDIUM' if rca_scores[c] > 0.4 else 'LOW' for c in all_features]
}).sort_values('RCA_Score', ascending=False)

# ۹. ذخیره در اکسل (تک فایل)
try:
    with pd.ExcelWriter(output_filename4, engine='openpyxl') as writer:
        # شیت ۱: نتایج تست ۳ برابر انحراف معیار روی داده‌های یک ماه اخیر
        cols_to_save = target_sensors + [f'Status_{c}' for c in target_sensors]
        df_fault[cols_to_save].to_excel(writer, sheet_name='Fault_Detection_Test')

        # شیت ۲: رتبه‌بندی ریشه عیب
        rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)

        # شیت ۳: ماتریس تغییرات روابط
        delta_pcorr.to_excel(writer, sheet_name='Partial_Corr_Delta')

    print(f"گزارش نهایی با موفقیت در این مسیر ذخیره شد:\n{output_filename4}")
    print("\n--- رتبه‌بندی ریشه عیب (RCA) ---")
    print(rca_summary.to_string(index=False))
    print("\n" + "="*70)
    print(f"🎯 کاندیدای اصلی Root Cause: {rca_summary.head(1)['Sensor'].tolist()}")

except Exception as e:
    print(f"خطا در ذخیره‌سازی: {e}")

✅ پردازش کامل شد. ستون‌های زیر اضافه شدند:
1. id (شماره ردیف)
2. date (تاریخ و زمان کامل)
3. RecordDate (تاریخ)
4. RecordTime (زمان)
✅ تعداد ردیف‌های نهایی (بدون NaN): 5675
✅ خروجی ذخیره شد.
